<a href="https://www.kaggle.com/code/hummern/wcag-2-2-violation-detection-with-strands-decider?scriptVersionId=355611548" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# WCAG 2.2 Violation Detection with Strands Decider

Train a Strands Decider model to classify HTML chunks as having WCAG 2.2 violations.

In [1]:
# Clone the WCAG fork
!git clone https://github.com/turbolego/strands-decider-wcag.git
%cd strands-decider-wcag

Cloning into 'strands-decider-wcag'...
remote: Enumerating objects: 420, done.
remote: Counting objects: 100% (93/93), done.
remote: Compressing objects: 100% (69/69), done.
remote: Total 420 (delta 43), reused 24 (delta 24), pack-reused 327 (from 1)
Receiving objects: 100% (420/420), 30.22 MiB | 22.13 MiB/s, done.
Resolving deltas: 100% (121/121), done.
/kaggle/working/strands-decider-wcag


In [2]:
# Install dependencies
!pip install -e ".[train]" -q
!pip install torch flash-linear-attention -q

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 15.0 MB/s eta 0:00:00
  Building editable for strands-decider (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.3/45.3 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 399.6/399.6 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 819.2/819.2 kB 25.4 MB/s eta 0:00:00


## Check GPU

In [3]:
!nvidia-smi

Tue Oct  6 02:24:58 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   51C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Check existing training data

In [4]:
!ls -la data/wcag/

total 80
drwxr-xr-x 2 root root  4096 Oct  6 02:24 .
drwxr-xr-x 6 root root  4096 Oct  6 02:24 ..
-rw-r--r-- 1 root root 53615 Oct  6 02:24 chunks.jsonl
-rw-r--r-- 1 root root 13890 Oct  6 02:24 train.jsonl


In [5]:
import json
with open('data/wcag/train.jsonl') as f:
    rows = [json.loads(l) for l in f]
print(f'{len(rows)} training rows')

# Count rules
from collections import Counter
rules = Counter()
for r in rows:
    for q in r['questions']:
        rules[q] += 1
print('Rules:', dict(rules.most_common(10)))

26 training rows
Rules: {'region': 6, 'landmark-unique': 4, 'tabindex': 4, 'landmark-contentinfo-is-top-level': 2, 'skip-link': 2, 'aria-allowed-attr': 1, 'aria-conditional-attr': 1, 'aria-prohibited-attr': 1, 'aria-valid-attr-value': 1, 'aria-valid-attr': 1}


## Stage 1: Build corpus (fast, CPU)

In [6]:
# Build the WCAG training corpus
!bash training/recipe_v7.sh build

############ build the v5 corpus ############
building ag_news ...
Generating train split: 100%|█| 120000/120000 [00:00<00:00, 507151.05 examples/s
Generating test split: 100%|█████| 7600/7600 [00:00<00:00, 721780.42 examples/s]
  6,000 examples
building banking77 ...
Generating test split: 100%|█████| 3080/3080 [00:00<00:00, 734921.85 examples/s]
  8,000 examples
building clinc150 ...
Generating train split: 100%|█| 15250/15250 [00:00<00:00, 1209475.96 examples/s]
Generating validation split: 100%|█| 3100/3100 [00:00<00:00, 707995.77 examples/
Generating test split: 100%|████| 5500/5500 [00:00<00:00, 1053748.95 examples/s]
  6,000 examples
building dbpedia ...
Generating train split: 100%|█| 560000/560000 [00:00<00:00, 749678.03 examples/s
Generating test split: 100%|███| 70000/70000 [00:00<00:00, 763634.11 examples/s]
  4,000 examples
building lang_id ...
Generating train split: 100%|██| 70000/70000 [00:00<00:00, 232710.49 examples/s]
Generating validation split: 100%|█| 10000/10000 

In [7]:
# Verify corpus was built
!wc -l data/train_v5.jsonl

100449 data/train_v5.jsonl


## Stage 2: Fetch upstream datasets (ContractNLI, MuSiQue)

In [8]:
!bash training/recipe.sh fetch

extracted ContractNLI and MuSiQue (full)


## Stage 3: Generate synthetic WCAG data (GPU — slow)

In [9]:
# Generate synthetic WCAG training data via Playwright + axe-core
# This creates additional training examples from real pages
!python scripts/generate_wcag_data.py \
  --url https://www.cdc.gov/ \
  --output data/wcag/synth_train.jsonl \
  --max-tokens 3000

Traceback (most recent call last):
  File "/kaggle/working/strands-decider-wcag/scripts/generate_wcag_data.py", line 22, in <module>
    from playwright.sync_api import sync_playwright
ModuleNotFoundError: No module named 'playwright'


## Stage 4: Multistep processing (GPU — slowest)

In [10]:
# Run multistep + synthetic generators
!bash training/recipe.sh generated

data/generated_v16p.jsonl: paraphrases attached to 2,121 of 2,148 rows from data/generated_v16.jsonl
data/para_pairs_v16_eval.jsonl: 339 eval rows from data/generated_v16_eval.jsonl, each with one paraphrase
data/generated_v18p.jsonl: paraphrases attached to 1,636 of 1,667 rows from data/generated_v18.jsonl
data/para_pairs_v18_eval.jsonl: 242 eval rows from data/generated_v18_eval.jsonl, each with one paraphrase


## Stage 5: Teacher model distillation (GPU — ~1h)

In [11]:
# Distill frozen Qwen3.5-4B teacher for multi-step rows
!bash training/recipe.sh teacher

data/multistep_v14.jsonl: FAILED
sha256sum: WARNING: 1 computed checksum did NOT match
stop: the files above do not match data/SHA256SUMS


## Stage 6: Parent model training (GPU — ~5h on P100)

In [12]:
# Train parent model
!export NGPU=2
!bash training/recipe.sh parent


data/multistep_v14.jsonl: FAILED
sha256sum: WARNING: 1 computed checksum did NOT match
stop: the files above do not match data/SHA256SUMS


## Stage 7: Replay parent for multi-step rows (GPU — ~40min)

In [13]:
# Replay parent distributions for multi-step rows
!bash training/recipe.sh replay

data/multistep_v14.jsonl: FAILED
sha256sum: WARNING: 1 computed checksum did NOT match
stop: the files above do not match data/SHA256SUMS


## Stage 8: Final model training (GPU — ~6h on P100, 1h on H100)

In [14]:
# Train final Strands Decider WCAG model
!export NGPU=2
!bash training/recipe.sh train


data/multistep_v14.jsonl: FAILED
sha256sum: WARNING: 1 computed checksum did NOT match
stop: the files above do not match data/SHA256SUMS


## Stage 9: Calibrate confidence scores (GPU — fast)

In [15]:
# Calibrate confidence scores on held-out sets
!bash training/recipe.sh calibrate

╭───────────────────── Traceback (most recent call last) ──────────────────────╮
│ /kaggle/working/strands-decider-wcag/src/strands_decider/cli.py:206 in       │
│ calibrate_cmd                                                                │
│                                                                              │
│   203 │   from .evaluate import calibrate_checkpoint, partition_examples,    │
│       sample_examples                                                        │
│   204 │                                                                      │
│   205 │   examples =                                                         │
│       sample_examples(partition_examples(list(read_jsonl(data)), split),     │
│       limit)                                                                 │
│ ❱ 206 │   result = calibrate_checkpoint(checkpoint, examples,                │
│       batch_size=batch_size)                                                 │
│   207 │   console.print(f"

## Stage 10: Evaluation (GPU — fast)

In [16]:
# Run evaluation on held-out sets
!bash training/recipe.sh eval

data/multistep_v14_eval.jsonl: FAILED
sha256sum: WARNING: 1 computed checksum did NOT match
stop: the files above do not match data/SHA256SUMS


## Serve the trained model

In [17]:
# Find the checkpoint
!find checkpoints/ -name '*.pt' -o -name '*.safetensors' | head -10

find: ‘checkpoints/’: No such file or directory


In [18]:
# Serve the model
!strands-decider serve checkpoints/hobson-2b-wcag --port 8000

serving checkpoints/hobson-2b-wcag on http://127.0.0.1:8000 (cuda)
╭───────────────────── Traceback (most recent call last) ──────────────────────╮
│ /usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_http.py:798   │
│ in hf_raise_for_status                                                       │
│                                                                              │
│    795 │   │   logger.debug("Failed to parse warning headers",               │
│        exc_info=True)                                                        │
│    796 │                                                                     │
│    797 │   try:                                                              │
│ ❱  798 │   │   response.raise_for_status()                                   │
│    799 │   except httpx.HTTPStatusError as e:                                │
│    800 │   │   if response.status_code // 100 == 3:                          │
│    801 │   │   │   return  # Do not rais

## Test the model

# Test the model

This cell attempts to test the served model. If the serve cell failed, this test will not run.
To test manually, run the serve cell and then execute:
```python
import requests, json
state = "<button style='width: 20px; height: 20px; padding: 0;'>Login</button>"
questions = {
    'wcag_2_5_8_target_size': {
        'type': 'noun',
        'instructions': 'Does this button meet the WCAG 2.2 minimum target size of 24x24 CSS pixels?'
    }
}
try:
    resp = requests.post('http://localhost:8000/v1/systemone',
        json={'state': state, 'questions': questions},
        headers={'content-type': 'application/json'})
    print(json.dumps(resp.json(), indent=2))
except Exception as e:
    print('Serve endpoint not running:', e)
```


## Kaggle Session Management Notes

In [19]:
# Kaggle has a 12-hour continuous session limit
# If training exceeds 12h, use checkpoint/resume:
# 1. The recipe saves checkpoints automatically in checkpoints/
# 2. To resume, set NGPU and re-run the failing stage
# 3. Use 'strands-decider train --config configs/train.yaml --resume checkpoints/hobson-2b-wcag'
# 4. Or use 'training/recipe.sh train' which checks for existing checkpoints